In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#fricative

tools_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ustools"
if tools_path not in sys.path:
    sys.path.append(tools_path)

from core import UltraSuiteCore
from transform_ultrasound import transform_ultrasound

input_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ultrasuite_subset"
file_fricative = "121_aud"

print(f"Loading data for fricative recording: {file_fricative}...")
data_fricative = UltraSuiteCore(directory=input_path, file_basename=file_fricative)

if data_fricative.ult.size == 0:
    raise FileNotFoundError(f"Could not load data. Ensure {file_fricative}.ult exists in {input_path}")


frame_idx_fricative = 175
raw_frame_fricative = data_fricative.ult[frame_idx_fricative]

transformed_data_fricative = transform_ultrasound(
    raw_frame_fricative, 
    num_scanlines=data_fricative.params['num_scanlines'],
    size_scanline=data_fricative.params['size_scanline'],
    angle=data_fricative.params['angle'],
    zero_offset=data_fricative.params['zero_offset'],
    pixels_per_mm=3
)

# Rotate exactly 90 degrees counterclockwise (k=1)
corrected_img_fricative = np.rot90(transformed_data_fricative[0], k=1)

# Compute 2D Fourier Transform on the CCW-rotated fricative image matrix
f_transform_fricative = np.fft.fft2(corrected_img_fricative)
f_shift_fricative = np.fft.fftshift(f_transform_fricative)
magnitude_2d_fricative = np.abs(f_shift_fricative)

# Calculate the center coordinates of the 2D Fourier spectrum
center_y_fricative, center_x_fricative = np.array(magnitude_2d_fricative.shape) // 2

# Create a coordinate grid containing distances from the center point
y_indices_fricative, x_indices_fricative = np.indices(magnitude_2d_fricative.shape)
r_grid_fricative = np.sqrt((x_indices_fricative - center_x_fricative)**2 + (y_indices_fricative - center_y_fricative)**2)
r_grid_fricative = r_grid_fricative.astype(int)

# Bin the pixels by their radial distance to find the average energy per frequency band
max_radius_fricative = r_grid_fricative.max()
radial_profile_fricative = np.zeros(max_radius_fricative + 1)

for r in range(max_radius_fricative + 1):
    mask_fricative = (r_grid_fricative == r)
    if mask_fricative.any():
        radial_profile_fricative[r] = magnitude_2d_fricative[mask_fricative].mean()

# Convert both spectrum elements to a log scale for visual readability
magnitude_2d_log_fricative = np.log(magnitude_2d_fricative + 1)
radial_profile_log_fricative = np.log(radial_profile_fricative + 1)

# --- FIGURE 1: Spatial Image and 2D Fourier Spectrum Side-by-Side ---
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4.2))

# Panel 1: Counterclockwise Rotated Wedge Image
axes1[0].imshow(corrected_img_fricative, cmap='gray', aspect='equal')
axes1[0].set_title(fr"Ultrasound /v/ in $\mathit{{everywhere}}$ (Frame {frame_idx_fricative})")
axes1[0].set_xlabel("Horizontal Pixels")
axes1[0].set_ylabel("Vertical Pixels")

# Panel 2: 2D Fourier Magnitude Spectrum
axes1[1].imshow(magnitude_2d_log_fricative, cmap='gray', aspect='equal')
axes1[1].set_title("/v/ 2D FFT Spectrum")
axes1[1].set_xlabel("Frequency (Horizontal)")
axes1[1].set_ylabel("Frequency (Vertical)")

# Maintain tight spacing between wide images
fig1.subplots_adjust(wspace=0.15, left=0.08, right=0.95, bottom=0.15, top=0.82)


# --- FIGURE 2: Standalone 1D Frequency Distribution Profile ---
plt.figure(figsize=(10, 4.5))

plt.plot(radial_profile_log_fricative, color='black', linewidth=2)
plt.title("Overall Fricative Frequency Distribution", fontsize=13, fontweight='semibold')
plt.xlabel("Spatial Frequency (Radial Pixel Distance from Center)", fontsize=11)
plt.ylabel("Log Magnitude (Energy)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()

# Render both windows/plots
plt.show()

In [ ]:
# This cell was used iterativly the identify the correct frame correspnding to the voice recording
#  Grab the correct ultrasound sampling rate (Frames Per Second)
f_rate = data.params['ult_fps']

# 2. Calculate the index for 9 seconds
target_time = 2.8
sec_index = int(target_time * f_rate)

# 3. Check your array dimensions
total_frames = data.ult.shape[0]

print(f"The ultrasound sampling rate is: {f_rate} frames per second")
print(f"Calculated index for {target_time} seconds: {sec_index}")
print(f"Total available frames in this file: {total_frames}")

if sec_index < total_frames:
    print(f"Success! Use frame_idx = {sec_index} in your script.")
else:
    print(f"⚠ Warning: This recording is only {total_frames / f_rate:.2f} seconds long. "
          f"Index {sec_index} is out of bounds. Using the last frame instead.")

In [ ]:
#r

tools_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ustools"
if tools_path not in sys.path:
    sys.path.append(tools_path)

from core import UltraSuiteCore
from transform_ultrasound import transform_ultrasound

input_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ultrasuite_subset"
file_r = "121_aud"

print(f"Loading data for vowel recording: {file_r}...")
data_r = UltraSuiteCore(directory=input_path, file_basename=file_r)

if data_r.ult.size == 0:
    raise FileNotFoundError(f"Could not load data. Ensure {file_vowel}.ult exists in {input_path}")

frame_idx_r = 228
raw_frame_r = data_r.ult[frame_idx_r]

transformed_data_r = transform_ultrasound(
    raw_frame_r, 
    num_scanlines=data_r.params['num_scanlines'],
    size_scanline=data_r.params['size_scanline'],
    angle=data_r.params['angle'],
    zero_offset=data_r.params['zero_offset'],
    pixels_per_mm=3
)

# Rotate exactly 90 degrees counterclockwise (k=1)
corrected_img_r = np.rot90(transformed_data_r[0], k=1)

# Compute 2D Fourier Transform on the CCW-rotated vowel image matrix
f_transform_r = np.fft.fft2(corrected_img_r)
f_shift_r = np.fft.fftshift(f_transform_r)
magnitude_2d_r = np.abs(f_shift_r)

# Calculate the center coordinates of the 2D Fourier spectrum
center_y_r, center_x_r = np.array(magnitude_2d_r.shape) // 2

# Create a coordinate grid containing distances from the center point
y_indices_r, x_indices_r = np.indices(magnitude_2d_r.shape)
r_grid_r = np.sqrt((x_indices_r - center_x_r)**2 + (y_indices_r - center_y_r)**2)
r_grid_r = r_grid_r.astype(int)

# Bin the pixels by their radial distance to find the average energy per frequency band
max_radius_r = r_grid_r.max()
radial_profile_r = np.zeros(max_radius_r + 1)

for r in range(max_radius_r + 1):
    mask_r = (r_grid_r == r)
    if mask_r.any():
        radial_profile_r[r] = magnitude_2d_r[mask_r].mean()

# Convert both spectrum elements to a log scale for visual readability
magnitude_2d_log_r = np.log(magnitude_2d_r + 1)
radial_profile_log_r = np.log(radial_profile_r + 1)


# --- FIGURE 1: Spatial Image and 2D Fourier Spectrum Side-by-Side ---
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4.2))

# Panel 1: Counterclockwise Rotated Wedge Image
axes1[0].imshow(corrected_img_r, cmap='gray', aspect='equal')
axes1[0].set_title(fr"Ultrasound /r/ in $\mathit{{problem}}$ (Frame {frame_idx_r})")
axes1[0].set_xlabel("Horizontal Pixels")
axes1[0].set_ylabel("Vertical Pixels")

# Panel 2: 2D Fourier Magnitude Spectrum
axes1[1].imshow(magnitude_2d_log_r, cmap='gray', aspect='equal')
axes1[1].set_title("/r/ 2D FFT Spectrum")
axes1[1].set_xlabel("Frequency (Horizontal)")
axes1[1].set_ylabel("Frequency (Vertical)")

# Maintain tight spacing between wide images
fig1.subplots_adjust(wspace=0.15, left=0.08, right=0.95, bottom=0.15, top=0.82)


# --- FIGURE 2: Standalone 1D Frequency Distribution Profile ---
plt.figure(figsize=(8, 4.5))

plt.plot(radial_profile_log_r, color='black', linewidth=2)
plt.title("Overall Vowel Frequency Distribution", fontsize=13, fontweight='semibold')
plt.xlabel("Spatial Frequency (Radial Pixel Distance from Center)", fontsize=11)
plt.ylabel("Log Magnitude (Energy)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()

# Render both windows/plots
plt.show()

In [ ]:

tools_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ustools"
if tools_path not in sys.path:
    sys.path.append(tools_path)

from core import UltraSuiteCore
from transform_ultrasound import transform_ultrasound

input_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ultrasuite_subset"
file_vowel = "121_aud"

print(f"Loading data for vowel recording: {file_vowel}...")
data_vowel = UltraSuiteCore(directory=input_path, file_basename=file_vowel)

if data_vowel.ult.size == 0:
    raise FileNotFoundError(f"Could not load data. Ensure {file_vowel}.ult exists in {input_path}")

frame_idx_vowel = 264
raw_frame_vowel = data_vowel.ult[frame_idx_vowel]

transformed_data_vowel = transform_ultrasound(
    raw_frame_vowel, 
    num_scanlines=data_vowel.params['num_scanlines'],
    size_scanline=data_vowel.params['size_scanline'],
    angle=data_vowel.params['angle'],
    zero_offset=data_vowel.params['zero_offset'],
    pixels_per_mm=3
)

# Rotate exactly 90 degrees counterclockwise (k=1)
corrected_img_vowel = np.rot90(transformed_data_vowel[0], k=1)

# Compute 2D Fourier Transform on the CCW-rotated vowel image matrix
f_transform_vowel = np.fft.fft2(corrected_img_vowel)
f_shift_vowel = np.fft.fftshift(f_transform_vowel)
magnitude_2d_vowel = np.abs(f_shift_vowel)

# Calculate the center coordinates of the 2D Fourier spectrum
center_y_vowel, center_x_vowel = np.array(magnitude_2d_vowel.shape) // 2

# Create a coordinate grid containing distances from the center point
y_indices_vowel, x_indices_vowel = np.indices(magnitude_2d_vowel.shape)
r_grid_vowel = np.sqrt((x_indices_vowel - center_x_vowel)**2 + (y_indices_vowel - center_y_vowel)**2)
r_grid_vowel = r_grid_vowel.astype(int)

# Bin the pixels by their radial distance to find the average energy per frequency band
max_radius_vowel = r_grid_vowel.max()
radial_profile_vowel = np.zeros(max_radius_vowel + 1)

for r in range(max_radius_vowel + 1):
    mask_vowel = (r_grid_vowel == r)
    if mask_vowel.any():
        radial_profile_vowel[r] = magnitude_2d_vowel[mask_vowel].mean()

# Convert both spectrum elements to a log scale for visual readability
magnitude_2d_log_vowel = np.log(magnitude_2d_vowel + 1)
radial_profile_log_vowel = np.log(radial_profile_vowel + 1)


# --- FIGURE 1: Spatial Image and 2D Fourier Spectrum Side-by-Side ---
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4.2))

# Panel 1: Counterclockwise Rotated Wedge Image
axes1[0].imshow(corrected_img_vowel, cmap='gray', aspect='equal')
axes1[0].set_title(fr"Ultrasound /a/ in $\mathit{{are}}$ (Frame {frame_idx_vowel})")
axes1[0].set_xlabel("Horizontal Pixels")
axes1[0].set_ylabel("Vertical Pixels")

# Panel 2: 2D Fourier Magnitude Spectrum
axes1[1].imshow(magnitude_2d_log_vowel, cmap='gray', aspect='equal')
axes1[1].set_title("/a/ 2D FFT Spectrum")
axes1[1].set_xlabel("Frequency (Horizontal)")
axes1[1].set_ylabel("Frequency (Vertical)")

# Maintain tight spacing between wide images
fig1.subplots_adjust(wspace=0.15, left=0.08, right=0.95, bottom=0.15, top=0.82)


# --- FIGURE 2: Standalone 1D Frequency Distribution Profile ---
plt.figure(figsize=(8, 4.5))

plt.plot(radial_profile_log_vowel, color='black', linewidth=2)
plt.title("Overall Vowel Frequency Distribution", fontsize=13, fontweight='semibold')
plt.xlabel("Spatial Frequency (Radial Pixel Distance from Center)", fontsize=11)
plt.ylabel("Log Magnitude (Energy)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()

# Render both windows/plots
plt.show()

In [ ]:

tools_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ustools"
if tools_path not in sys.path:
    sys.path.append(tools_path)

from core import UltraSuiteCore
from transform_ultrasound import transform_ultrasound

#fresh data loading
input_path = "/Users/ebennett/Library/CloudStorage/Dropbox-EMI/Evelyn Bennett/misc_project/m70/m70project/ultrasuite-tools-master/ultrasuite_subset"
file_base = "121_aud"

print("Freshly initializing UltraSuite data class for tri-phonetic matrix...")
fresh_data = UltraSuiteCore(directory=input_path, file_basename=file_base)

if fresh_data.ult.size == 0:
    raise FileNotFoundError(f"Ensure {file_base}.ult exists in {input_path}")

# spectrum isolation fxn
def get_highpass_spectrum(frame_idx, sigma=8.0):
    """Loads a frame, applies geometric correction, rotates CCW, and high-pass filters the 2D FFT."""
    raw_frame = fresh_data.ult[frame_idx]
    transformed = transform_ultrasound(
        raw_frame, 
        num_scanlines=fresh_data.params['num_scanlines'],
        size_scanline=fresh_data.params['size_scanline'],
        angle=fresh_data.params['angle'],
        zero_offset=fresh_data.params['zero_offset'],
        pixels_per_mm=3
    )
    # Rotate 90 degrees counterclockwise
    img_rotated = np.rot90(transformed[0], k=1)
    
    # 2D Fourier Transform
    fft_shifted = np.fft.fftshift(np.fft.fft2(img_rotated))
    mag_log = np.log(np.abs(fft_shifted) + 1)
    
    # Apply Gaussian High-Pass Filter
    lowpass = ndimage.gaussian_filter(mag_log, sigma=sigma)
    hp_mag = np.clip(mag_log - lowpass, 0, None)
    
    return hp_mag

# Extract filtered spectrums for all three targets
hp_vowel     = get_highpass_spectrum(264)
hp_rhotic    = get_highpass_spectrum(228)
hp_fricative = get_highpass_spectrum(175)

center_y, center_x = np.array(hp_vowel.shape) // 2
y_indices, x_indices = np.indices(hp_vowel.shape)
r_grid = np.sqrt((x_indices - center_x)**2 + (y_indices - center_y)**2).astype(int)
max_radius = r_grid.max()

def get_normalized_radial_profile(hp_matrix):
    """Bins 2D spectrum into a 1D radial vector, normalized as a probability distribution."""
    profile = np.zeros(max_radius + 1)
    for r in range(max_radius + 1):
        mask = (r_grid == r)
        if mask.any():
            profile[r] = hp_matrix[mask].mean()
            
    # Normalize profile to sum to 1, adding epsilon to stabilize KL Divergence
    p = profile / (np.sum(profile) + 1e-10)
    p += 1e-10
    return p / np.sum(p)

p_vowel = get_normalized_radial_profile(hp_vowel)
p_rhotic = get_normalized_radial_profile(hp_rhotic)
p_fricative = get_normalized_radial_profile(hp_fricative)

def compute_pairwise_metrics(matrixA, matrixB, profileA, profileB):
    """Computes normalized 2D SSIM and 1D KL Divergence between two sets."""
    normA = (matrixA - matrixA.min()) / (matrixA.max() - matrixA.min() + 1e-10)
    normB = (matrix2 - matrix2.min()) / (matrix2.max() - matrix2.min() + 1e-10) if 'matrix2' in locals() else (matrixB - matrixB.min()) / (matrixB.max() - matrixB.min() + 1e-10)
    
    ssim_val = ssim(normA, normB, data_range=1.0)
    kl_val = np.sum(rel_entr(profileA, profileB))
    return ssim_val, kl_val

# Execute cross-comparisons
ssim_vf, kl_vf = compute_pairwise_metrics(hp_vowel, hp_fricative, p_vowel, p_fricative)
ssim_vr, kl_vr = compute_pairwise_metrics(hp_vowel, hp_rhotic, p_vowel, p_rhotic)
ssim_rf, kl_rf = compute_pairwise_metrics(hp_rhotic, hp_fricative, p_rhotic, p_fricative)

# print summary matrix
print("\n" + "="*65)
print("          TRI-PHONETIC SPATIAL FREQUENCY COMPARISON MATRIX")
print("="*65)
print(f"Vowel [ɑ] vs Fricative [v]:   SSIM = {ssim_vf:.4f}  |  KL Divergence = {kl_vf:.4f}")
print(f"Vowel [ɑ] vs Rhotic [r]:      SSIM = {ssim_vr:.4f}  |  KL Divergence = {kl_vr:.4f}")
print(f"Rhotic [r] vs Fricative [v]:  SSIM = {ssim_rf:.4f}  |  KL Divergence = {kl_rf:.4f}")
print("="*65 + "\n")

plt.figure(figsize=(10, 5))
plt.plot(hp_vowel[center_y, center_x:], label='Vowel [ɑ] (Frame 264)', color='blue', linewidth=2)
plt.plot(hp_rhotic[center_y, center_x:], label='Rhotic [r] (Frame 228)', color='green', linewidth=2, linestyle='-.')
plt.plot(hp_fricative[center_y, center_x:], label='Fricative [v] (Frame 175)', color='red', linewidth=2, linestyle='--')

plt.title("Filtered Spatial Frequency Spectrum Across the Articulatory Continuum", fontsize=13, fontweight='semibold')
plt.xlabel("Spatial Frequency (Radial Pixel Distance from Center)", fontsize=11)
plt.ylabel("Filtered Log Magnitude Energy", fontsize=11)
plt.legend(frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()